# GLiNER: первый NER-пайплайн на синтетических данных

Цель notebook — на маленьком безопасном примере пройти весь путь: **разметка → валидация → обучение → консервативная выдача → precision-метрики**. Это учебный сценарий, а не доказательство качества: для решения о запуске нужен отдельный вручную проверенный golden test.

> Техническая оговорка: этот notebook использует API новой библиотеки `gliner2`. Если в ТЗ под «GLiNER v2» имеется в виду старая библиотека `gliner` с checkpoint `*-v2.x`, нужно заменить только секции «обучение» и «инференс», но формат данных, разметка, метрики и постобработка сохраняются.

## 0. Установка

Один раз в терминале из корня репозитория: `make install-training`. Затем выберите kernel созданного `uv`-окружения. Notebook не создаёт и не отправляет реальные данные.

In [ ]:
import random
from collections import defaultdict
from pathlib import Path

from bank_ner.evaluation import exact_span_metrics, presence_metrics
from bank_ner.postprocess import filter_entities, load_blocked_persons
from bank_ner.schema import Entity, Example, read_jsonl

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "data" / "synthetic" / "demo.jsonl"
ARTIFACTS_DIR = ROOT / "artifacts" / "synthetic_gliner2"
LABELS = ["PERSON", "ORG", "CONTACT_ROLE"]
LABEL_DESCRIPTIONS = {
    "PERSON": "имя, фамилия или ФИО реального частного получателя; не знаменитость и не блюдо",
    "ORG": "получатель платежа, поставщик услуги, сервис или счёт",
    "CONTACT_ROLE": "роль личного контакта: мама, папа, сестра, тренер и формы",
}

assert DATA_PATH.exists(), f"Откройте notebook из проекта: {DATA_PATH} не найден"
print(ROOT)

## 1. Прочитать и проверить разметку

Схема JSONL всегда хранит точные символьные позиции `[start:end)`. Валидатор немедленно остановит пайплайн, если LLM или человек вернули неверные границы.

In [ ]:
examples = read_jsonl(DATA_PATH)
for example in examples:
    spans = [(e.text, e.label, e.start, e.end) for e in example.entities]
    print(example.id, "|", example.text, "|", spans)

negative_count = sum(not example.entities for example in examples)
print(f"Всего: {len(examples)}; негативных примеров: {negative_count}")

## 2. Как выглядит разметка LLM

При переносе в промышленный контур возьмите `ANNOTATION_JSON_SCHEMA` и `system_prompt` из `bank_ner.llm_contract`. В запрос LLM передаётся исходная строка и JSON Schema, ответ строго валидируется. Только `high`-уверенность может стать кандидатом для train; остальные записи идут в ручную проверку.

In [ ]:
from bank_ner.llm_contract import ANNOTATION_JSON_SCHEMA, system_prompt, validate_llm_annotation

query = "Переведи Антону в Мосэнергосбыт"
llm_response = {
    "entities": [
        {"start": 9, "end": 15, "label": "PERSON", "text": "Антону", "confidence": "high"},
        {"start": 18, "end": 31, "label": "ORG", "text": "Мосэнергосбыт", "confidence": "high"},
    ]
}
accepted, needs_review = validate_llm_annotation(llm_response, query)
print("В train-кандидаты:", accepted)
print("На проверку:", needs_review)
print(system_prompt(["Пушкин", "Юрий Гагарин"])[:350] + "…")
assert ANNOTATION_JSON_SCHEMA["schema"]["properties"]["entities"]["type"] == "array"

## 3. Разделить данные без утечки

На реальных данных split делается до обучения и сохраняется. Не допускайте, чтобы одинаковые или почти одинаковые варианты ФИО/организации попадали и в train, и в validation. Здесь мини-датасет слишком мал, поэтому разбиение только демонстрационное.

In [ ]:
rng = random.Random(42)
shuffled = examples.copy()
rng.shuffle(shuffled)
split_at = max(1, int(len(shuffled) * 0.8))
train_examples, validation_examples = shuffled[:split_at], shuffled[split_at:]
print("train:", [example.id for example in train_examples])
print("validation:", [example.id for example in validation_examples])
print("В production: golden test set не участвует ни в этом разбиении, ни в подборе порога.")

## 4. Подготовить формат GLiNER2

Библиотека принимает текст и словарь `метка → список поверхностей`. Наш внутренний формат со спанами остаётся источником истины: он нужен для валидации и честной оценки границ.

In [ ]:
def to_gliner2_example(example: Example):
    from gliner2.training.data import InputExample

    grouped = defaultdict(list)
    for entity in example.entities:
        grouped[entity.label].append(entity.text)
    return InputExample(
        text=example.text,
        entities=dict(grouped),
        entity_descriptions=LABEL_DESCRIPTIONS,
    )


# Не запускайте эту ячейку до `make install-training`.
# train_gliner2 = [to_gliner2_example(example) for example in train_examples]
# validation_gliner2 = [to_gliner2_example(example) for example in validation_examples]

## 5. Загрузить и дообучить модель

Это первый пробный запуск. На 6 синтетических строках модель **не станет полезной**; цель — убедиться, что окружение, форматы и сохранение артефакта работают. Для реального обучения начать с малого числа эпох и следить именно за precision на validation.

In [ ]:
# Выполните после подтверждения, что ТЗ требует именно пакета gliner2.
# import torch
# from gliner2 import GLiNER2
# from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig
#
# MODEL_ID = 'fastino/gliner2-base-v1'  # замените на одобренный внутренний checkpoint при наличии
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model = GLiNER2.from_pretrained(MODEL_ID, map_location=device)
# config = TrainingConfig(
#     output_dir=str(ARTIFACTS_DIR),
#     experiment_name='synthetic-bank-ner',
#     num_epochs=3,
#     batch_size=4,
#     encoder_lr=1e-5,
#     task_lr=5e-4,
#     warmup_ratio=0.1,
#     fp16=(device == 'cuda'),
#     eval_strategy='epoch',
#     save_best=True,
#     early_stopping=True,
#     early_stopping_patience=2,
# )
# trainer = GLiNER2Trainer(model, config)
# trainer.train(train_data=train_gliner2, val_data=validation_gliner2)

## 6. Precision-first postprocessing

Модельные предсказания проходят через: (1) отдельные пороги по меткам, выбранные на validation; (2) проверенный список исключений знаменитостей. Блэклист — защитный слой, но не замена негативным примерам и golden test.

In [ ]:
blocked = load_blocked_persons(ROOT / "configs" / "blocked_persons.txt")
thresholds = {"PERSON": 0.92, "ORG": 0.90, "CONTACT_ROLE": 0.95}  # пример, не production-значения

# Такой адаптированный список должен вернуть слой интеграции модели.
# У GLiNER2 сначала посмотрите сырой ответ extract_entities(
#     ..., include_spans=True, include_confidence=True
# ),
# затем приведите его к Entity в одном адаптере, а не в бизнес-коде.
raw_prediction = [
    Entity(0, 6, "PERSON", "Пушкин"),
    Entity(18, 31, "ORG", "Мосэнергосбыт"),
]
scores = {(0, 6, "PERSON"): 0.99, (18, 31, "ORG"): 0.96}
kept = filter_entities(
    raw_prediction, blocked_persons=blocked, thresholds=thresholds, scores=scores
)
print("После защиты:", kept)

## 7. Считать метрики

Exact-span precision — основная метрика: неверная граница считается ошибкой. Query-level presence дополнительно показывает, насколько часто downstream получит ложный сигнал «в запросе есть сущность».

In [ ]:
gold = {example.id: list(example.entities) for example in examples}
# Демонстрация: модель нашла всё, кроме одного ORG,
# и ошибочно нашла PERSON в негативе.
predicted = {example.id: list(example.entities) for example in examples}
predicted["syn-002"] = []
predicted["syn-005"] = [Entity(9, 16, "PERSON", "Пушкину")]

span_report = exact_span_metrics(gold, predicted)
presence_report = presence_metrics(gold, predicted)
span_report, presence_report

## Что переносить в production

1. Создайте неизменяемый `gold_test.jsonl` (500–1 000 вручную проверенных запросов).
2. Исторические 0/1-флаги используйте для стратифицированного отбора, но не как truth.
3. LLM размечает только спаны по контракту выше; medium/low — в review queue.
4. После обучения переберите пороги на validation, выбрав лучший допустимый precision; не подбирайте их на golden test.
5. Сохраните вместе с моделью: commit данных, пороги, блоклист, точный checkpoint, метрики и дату.
6. Для регулярного переобучения добавляйте подтверждённые FP/FN в train и повторяйте оценку на неизменяемом golden test.